# Load deps

In [ ]:
# ! pip install -q torcheval

In [ ]:
# # if src modules imported
# from google.colab import drive
# drive.mount('/content/drive')
import sys
app_path = '/content/drive/MyDrive/Projects/miniSD'
sys.path.append(app_path)

In [ ]:
import shutil, torch, os
import matplotlib as mpl
import torchvision.transforms as T
import torchvision.transforms.functional as TF
import torch.nn.functional as F
from pathlib import Path
from glob import glob
from functools import partial
from torch import nn, optim
from torch.optim import lr_scheduler
from torchvision.io import read_image,ImageReadMode

from src.utils import set_seed, noop, urlsave
from src.datasets import DataLoaders, show_images
from src.training import get_dls
from src.learner import DeviceCB, ProgressCB, MetricsCB, Learner, to_cpu
from src.accel import MixedPrecision
from src.sgd import BatchSchedCB
from src.augment import rand_erase
from src.init import init_weights, clean_mem, GeneralRelu
from src.resnet import ResBlock

# Config

In [ ]:
# os.environ['CUDA_VISIBLE_DEVICES']='2' --> not needed if we have a single GPU
torch.set_printoptions(precision=5, linewidth=140, sci_mode=False)
mpl.rcParams['figure.dpi'] = 70
# set_seed(42)
bs = 512
data_path = Path('artifacts/data')
data_path.mkdir(exist_ok=True, parents=True)
models_path = Path("artifacts/models")
models_path.mkdir(exist_ok=True, parents=True)

# Data processing

In [ ]:
tin_path = data_path/'tiny-imagenet-200'
url = 'http://cs231n.stanford.edu/tiny-imagenet-200.zip'
if not tin_path.exists():
    path_zip = urlsave(url, data_path)
    shutil.unpack_archive(path_zip, data_path)

In [ ]:
all_img_list = [
    read_image(img, mode=ImageReadMode.RGB)/255
    for img in glob(str(tin_path/'train'/'**/*.JPEG'), recursive=True)
]
all_img_pt = torch.stack(all_img_list)

xmean = all_img_pt.mean(dim=(0,2,3))
xstd = all_img_pt.std(dim=(0,2,3))
print(f"xmean: {xmean} | xstd: {xstd}")

del all_img_list, all_img_pt

In [ ]:
# input(x) is the image --> resized from 64x64 to 32x32 --> interpolated to 64x64
# output(x) is the image
# for reconstruction tasks such as superres, these transformation should be applied on both input and output
# this is why we put this in the TinyDS class
# if overfitting is a problem, add trivial augment here
tfms = nn.Sequential(T.Pad(8), T.RandomCrop(64), T.RandomHorizontalFlip())

class TinyDS:
    def __init__(self, path):
        self.path = Path(path)
        self.files = glob(str(path/'**/*.JPEG'), recursive=True)
    def __len__(self): return len(self.files)
    def __getitem__(self, i):
        img = read_image(self.files[i], mode=ImageReadMode.RGB)/255
        return tfms((img-xmean[:,None,None])/xstd[:,None,None])

class TfmDS:
    def __init__(self, ds, tfmx=noop, tfmy=noop): self.ds,self.tfmx,self.tfmy = ds,tfmx,tfmy
    def __len__(self): return len(self.ds)
    def __getitem__(self, i):
        item = self.ds[i]
        return self.tfmx(item),self.tfmy(item)

def denorm(x): return (x*xstd[:,None,None]+xmean[:,None,None]).clamp(0,1)

In [ ]:
# these item-wise transformation are only applied to x (input)
def tfmx(x, erase=True):
    x = TF.resize(x, (32,32))[None]
    x = F.interpolate(x, scale_factor=2)
    if erase: x = rand_erase(x)
    return x[0]

tds = TinyDS(tin_path/'train')
vds = TinyDS(tin_path/'val')

tfm_tds = TfmDS(tds, tfmx)
tfm_vds = TfmDS(vds, partial(tfmx, erase=False))

dls = DataLoaders(*get_dls(tfm_tds, tfm_vds, bs=bs, num_workers=os.cpu_count()))

del tds, vds, tfm_tds, tfm_vds

In [ ]:
xb,yb = next(iter(dls.train))
show_images(denorm(xb[:4]), imsize=2.5);
show_images(denorm(yb[:4]), imsize=2.5);

# Denoising autoencoder

In [ ]:
act_gr = partial(GeneralRelu, leak=0.1, sub=0.4)
iw = partial(init_weights, leaky=0.1)
def up_block(ni, nf, ks=3, act=act_gr, norm=None):
    return nn.Sequential(
        nn.UpsamplingNearest2d(scale_factor=2), # this is actually the inverse of stride=2
        ResBlock(ni, nf, ks=ks, act=act, norm=norm)
    )

# this uses the original resblock arch, not the new one with pure skip connection
# TODO: what happens if we use the new arch?
def get_model(act=act_gr, nfs=(32,64,128,256,512,1024), norm=nn.BatchNorm2d, drop=0.1):
    layers = [ResBlock(3, nfs[0], ks=5, stride=1, act=act, norm=norm)]
    layers += [ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2) for i in range(len(nfs)-1)]
    layers += [up_block(nfs[i], nfs[i-1], act=act, norm=norm) for i in range(len(nfs)-1,0,-1)]
    layers += [ResBlock(nfs[0], 3, act=nn.Identity, norm=norm)]
    # the output image needs 3 channels
    return nn.Sequential(*layers).apply(iw)

# see the summary below to view the model arch

In [ ]:
metrics = MetricsCB()
cbs = [DeviceCB(), metrics, ProgressCB(plot=True), MixedPrecision()]
lr_cbs = [DeviceCB(), ProgressCB(), MixedPrecision()]
opt_func = partial(optim.AdamW, eps=1e-5)

In [ ]:
init_learner = Learner(get_model().apply(iw), dls, F.mse_loss, cbs=lr_cbs, opt_func=opt_func)
init_learner.lr_find(start_lr=1e-5, gamma=1.2)

In [ ]:
init_learner.summary()

In [ ]:
epochs = 5
lr = 1e-2
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
learn = Learner(get_model().apply(iw), dls, F.mse_loss, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

In [ ]:
learn.fit(epochs)

In [ ]:
p,t,inp = learn.capture_preds(inps=True);
show_images(denorm(inp[:9]), imsize=2);
show_images(denorm(p[:9]), imsize=2);

- The model performs very poorly!!!

# Unet

- One major difference introduced in unet [paper](https://arxiv.org/abs/1505.04597) is the formation of skip connections
    - rather than connecting subsequent layers, we connect 0 to nth, 1 to (n-1)th and so on.
    - can be experimented with and change the exact start and end point of skip connections to see the performance changes
    - another possible improvement for unet is to use more complex CNNs at the depth of the Unet
        - it adds less computation cost because of the downsampled image dimension
        - it can improve the performance by increasing the model's complexity

In [ ]:
del(learn)
clean_mem()

## Simple Unet arch

In [ ]:
class TinyUnet(nn.Module):
    def __init__(self, act=act_gr, nfs=(32,64,128,256,512,1024), norm=nn.BatchNorm2d):
        super().__init__()
        self.start = ResBlock(3, nfs[0], stride=1, act=act, norm=norm)

        # the down path
        self.dn = nn.ModuleList([ResBlock(nfs[i], nfs[i+1], act=act, norm=norm, stride=2)
                                 for i in range(len(nfs)-1)])
        
        # the up path
        self.up = nn.ModuleList([up_block(nfs[i], nfs[i-1], act=act, norm=norm)
                                 for i in range(len(nfs)-1,0,-1)])
        self.up += [ResBlock(nfs[0], 3, act=act, norm=norm)]

        # this is applied on the last summation
        self.end = ResBlock(3, 3, act=nn.Identity, norm=norm)

    def __iter__(self):
        yield self.start
        yield from self.dn
        yield from self.up
        yield self.end

    def forward(self, x):
        layers = []
        layers.append(x)
        x = self.start(x)
        for l in self.dn:
            layers.append(x)
            x = l(x)
        n = len(layers)
        for i,l in enumerate(self.up):
            if i!=0: x += layers[n-i]
            # instead of concatentation used in the paper, we use summation
            x = l(x)
        return self.end(x+layers[0])


def zero_wgts(l):
    with torch.no_grad():
        l.weight.zero_()
        l.bias.zero_()

In [ ]:
model = TinyUnet()
last_res = model.up[-1]

# this makes the input equal to output at the beginning
zero_wgts(last_res.convs[-1][-1])
zero_wgts(last_res.idconv[0])
zero_wgts(model.end.convs[-1][-1])

init_unet_learner = Learner(model, dls, F.mse_loss, cbs=lr_cbs, opt_func=opt_func)
init_unet_learner.lr_find(start_lr=1e-5, gamma=1.2)

## Show model DAG

In [ ]:
from src.conv import show_model_dag
show_model_dag(model, xb[:1])

## Train

In [ ]:
model = TinyUnet()
last_res = model.up[-1]
zero_wgts(last_res.convs[-1][-1])
zero_wgts(last_res.idconv[0])
zero_wgts(model.end.convs[-1][-1])

epochs = 10
lr = 4e-3
tmax = epochs * len(dls.train)
sched = partial(lr_scheduler.OneCycleLR, max_lr=lr, total_steps=tmax)
xtra = [BatchSchedCB(sched)]
learn = Learner(model, dls, F.mse_loss, lr=lr, cbs=cbs+xtra, opt_func=opt_func)

In [ ]:
learn.fit(epochs)

In [ ]:
p,t,inp = learn.capture_preds(inps=True);
show_images(denorm(inp[:9]), imsize=2);
show_images(denorm(p[:9]), imsize=2);
show_images(denorm(t[:9]), imsize=2);

- not able to reconstruct images very well
- but much better than the denoising auto encoder